# Processamento de Linguagem Natural
## Etapa Prática 1: Coleta e preparação de dados

**Equipe:** Marcos Borgert e Vitor Gustavo Hornburg  
**Tema:** Notícias sobre ações do mercado financeiro brasileiro  
**Fonte:** Status Invest

### Objetivo

Coletar notícias sobre ações brasileiras e preparar os textos para uso em tarefas de PLN.

O notebook faz:
- coleta das notícias;
- organização da base;
- limpeza dos textos;
- tokenização;
- normalização;
- remoção de stopwords;
- lematização;
- stemming;
- exportação dos arquivos finais.

## 1. Base de dados

A base é formada por notícias publicadas no Status Invest.

Foram escolhidas notícias do mercado financeiro porque elas possuem bastante conteúdo textual e podem ser usadas depois em tarefas como classificação, análise de sentimento, busca por empresa e resumo de notícias.

O recorte considera alguns ativos conhecidos da B3:
`PETR4`, `VALE3`, `ITUB4`, `WEGE3` e `BBAS3`.

In [ ]:
%pip -q install requests beautifulsoup4 pandas nltk spacy lxml certifi

In [ ]:
import json
import re
import time
import html
import subprocess
import sys
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

import spacy
import certifi

pd.set_option("display.max_colwidth", 150)

## 2. Configuração da coleta

O código abaixo define o site, os ativos usados no recorte e a quantidade de páginas que serão percorridas.

A pausa entre as requisições evita fazer muitos acessos em sequência.

In [ ]:
BASE_URL = "https://statusinvest.com.br"
NEWS_URL = "https://statusinvest.com.br/noticias/"

TICKERS_ALVO = ["PETR4", "VALE3", "ITUB4", "WEGE3", "BBAS3"]

# Aumente esse valor para ampliar a base.
PAGINAS_PARA_COLETAR = 50

PAUSA_SEGUNDOS = 1.0

HEADERS = {
    "User-Agent": "Mozilla/5.0 AppleWebKit/537.36 Chrome/140 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
}

def criar_sessao():
    sessao = requests.Session()

    retry = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    sessao.mount("https://", HTTPAdapter(max_retries=retry))
    sessao.headers.update(HEADERS)

    return sessao

sessao = criar_sessao()

## 3. Coleta dos links

Primeiro são coletados os links das páginas de notícias.

O código percorre a paginação e guarda apenas os links que parecem apontar para uma notícia.

In [ ]:
def eh_url_de_artigo(url):
    if not url.startswith(f"{BASE_URL}/noticias/"):
        return False

    caminho = url.replace(f"{BASE_URL}/noticias/", "", 1).strip("/")

    if not caminho:
        return False

    if caminho.startswith("page/") or caminho.startswith("c/"):
        return False

    return True


def coletar_links_noticias(max_paginas=50, pausa=1.0):
    links = set()

    for pagina in range(1, max_paginas + 1):
        url = NEWS_URL if pagina == 1 else f"{NEWS_URL}page/{pagina}/"

        print(f"Página {pagina}: {url}")

        resposta = sessao.get(url, timeout=30)
        resposta.raise_for_status()

        soup = BeautifulSoup(resposta.text, "lxml")

        antes = len(links)

        for a in soup.select("a[href]"):
            href = a.get("href", "").strip()
            url_completa = urljoin(BASE_URL, href).split("#")[0].split("?")[0]

            if eh_url_de_artigo(url_completa):
                links.add(url_completa)

        novos = len(links) - antes
        print(f"Novos links: {novos} | Total: {len(links)}")

        time.sleep(pausa)

    return sorted(links)


links_noticias = coletar_links_noticias(
    max_paginas=PAGINAS_PARA_COLETAR,
    pausa=PAUSA_SEGUNDOS
)

print("Total de links encontrados:", len(links_noticias))
links_noticias[:10]

## 4. Extração das notícias

Depois de coletar os links, o próximo passo é abrir cada notícia e tentar obter:
- título;
- data;
- autor;
- texto da notícia.

Quando possível, o código usa dados estruturados da própria página.

In [ ]:
def iterar_jsonld(soup):
    for script in soup.find_all("script", type="application/ld+json"):
        conteudo = script.string or script.get_text(strip=True)

        if not conteudo:
            continue

        try:
            obj = json.loads(conteudo)
        except Exception:
            continue

        if isinstance(obj, list):
            itens = obj
        elif isinstance(obj, dict) and isinstance(obj.get("@graph"), list):
            itens = obj["@graph"]
        else:
            itens = [obj]

        for item in itens:
            if isinstance(item, dict):
                yield item


def primeiro_texto(soup, seletores):
    for seletor in seletores:
        elemento = soup.select_one(seletor)

        if elemento:
            texto = elemento.get_text(" ", strip=True)

            if texto:
                return texto

    return None


def extrair_tickers(texto):
    if not isinstance(texto, str):
        return []

    encontrados = re.findall(r"\b[A-Z]{4}\d{1,2}\b", texto.upper())

    return sorted(set(encontrados))


def extrair_noticia(url):
    resposta = sessao.get(url, timeout=30)
    resposta.raise_for_status()

    soup = BeautifulSoup(resposta.text, "lxml")

    dados = {
        "url": url,
        "titulo": None,
        "data_publicacao": None,
        "autor": None,
        "texto_bruto": None,
    }

    for item in iterar_jsonld(soup):
        tipo = item.get("@type")
        tipos = tipo if isinstance(tipo, list) else [tipo]

        if any(t in {"Article", "NewsArticle", "BlogPosting"} for t in tipos):
            dados["titulo"] = item.get("headline") or item.get("name")
            dados["data_publicacao"] = item.get("datePublished")

            autor = item.get("author")

            if isinstance(autor, dict):
                dados["autor"] = autor.get("name")
            elif isinstance(autor, list):
                nomes = [
                    a.get("name") if isinstance(a, dict) else str(a)
                    for a in autor
                ]
                dados["autor"] = ", ".join([n for n in nomes if n])
            elif autor:
                dados["autor"] = str(autor)

            dados["texto_bruto"] = item.get("articleBody")
            break

    if not dados["titulo"]:
        og = soup.select_one('meta[property="og:title"]')

        if og:
            dados["titulo"] = og.get("content")

    if not dados["titulo"]:
        dados["titulo"] = primeiro_texto(soup, ["h1", "article h1"])

    if not dados["data_publicacao"]:
        tag_time = soup.select_one("time[datetime]")

        if tag_time:
            dados["data_publicacao"] = tag_time.get("datetime")

    if not dados["autor"]:
        meta_autor = soup.select_one('meta[name="author"]')

        if meta_autor:
            dados["autor"] = meta_autor.get("content")

    if not dados["texto_bruto"]:
        bloco = None

        for seletor in [
            "article",
            ".post-content",
            ".entry-content",
            ".article-content",
            ".content",
        ]:
            bloco = soup.select_one(seletor)

            if bloco:
                break

        if bloco:
            for lixo in bloco.select(
                "script, style, nav, footer, aside, form, button, "
                ".share, .social, .ads, .related"
            ):
                lixo.decompose()

            paragrafos = [
                p.get_text(" ", strip=True)
                for p in bloco.find_all("p")
                if p.get_text(" ", strip=True)
            ]

            dados["texto_bruto"] = (
                "\n".join(paragrafos)
                if paragrafos
                else bloco.get_text(" ", strip=True)
            )

    return dados

In [ ]:
registros = []

for i, url in enumerate(links_noticias, start=1):
    try:
        noticia = extrair_noticia(url)
        registros.append(noticia)

        print(f"[{i}/{len(links_noticias)}] OK: {noticia.get('titulo')}")

    except Exception as erro:
        print(f"[{i}/{len(links_noticias)}] Erro: {erro}")

    time.sleep(PAUSA_SEGUNDOS)

df_bruto = pd.DataFrame(registros)

df_bruto.head()

## 5. Identificação dos ativos

Nesta etapa são procurados tickers dentro do título e do texto da notícia.

Depois disso, ficam apenas as notícias que citam pelo menos um dos ativos escolhidos.

In [ ]:
def tickers_da_noticia(linha):
    texto = f"{linha.get('titulo') or ''} {linha.get('texto_bruto') or ''}"

    return extrair_tickers(texto)


df_bruto["tickers_encontrados"] = df_bruto.apply(
    tickers_da_noticia,
    axis=1
)

alvos = set(TICKERS_ALVO)

df = df_bruto[
    df_bruto["tickers_encontrados"].apply(
        lambda lista: bool(alvos.intersection(lista))
    )
].copy()

df.reset_index(drop=True, inplace=True)

print("Notícias coletadas:", len(df_bruto))
print("Notícias usadas no recorte:", len(df))

df[[
    "titulo",
    "tickers_encontrados",
    "data_publicacao",
    "url"
]].head(10)

## 6. Dicionário de dados

| Campo | Descrição |
|---|---|
| `url` | endereço da notícia |
| `titulo` | título da notícia |
| `data_publicacao` | data de publicação |
| `autor` | autor ou redação |
| `texto_bruto` | texto original coletado |
| `tickers_encontrados` | tickers encontrados no título e no texto |
| `texto_limpo` | texto depois da limpeza |
| `tokens` | palavras e sinais separados pelo tokenizador |
| `tokens_normalizados` | tokens em minúsculas e somente alfabéticos |
| `tokens_sem_stopwords` | tokens depois da remoção de stopwords |
| `lemas` | palavras reduzidas à forma canônica |
| `stems` | radicais gerados pelo stemming |

## 7. Conferência da base

Aqui são verificados:
- quantidade de registros;
- valores vazios;
- URLs repetidas;
- títulos repetidos;
- quantidade de notícias por ticker.

In [ ]:
print("Dimensão da base:", df.shape)

print("\nValores ausentes:")
display(df.isna().sum().to_frame("quantidade"))

print("\nURLs duplicadas:", df["url"].duplicated().sum())
print("Títulos duplicados:", df["titulo"].duplicated().sum())

In [ ]:
distribuicao_tickers = (
    df[["tickers_encontrados"]]
    .explode("tickers_encontrados")
    .query("tickers_encontrados in @TICKERS_ALVO")
    ["tickers_encontrados"]
    .value_counts()
)

display(distribuicao_tickers.to_frame("quantidade"))

## 8. Limpeza dos textos

A coluna `texto_bruto` é mantida.

A limpeza cria uma nova coluna e remove:
- URLs;
- espaços repetidos;
- quebras de linha;
- marcações como `H2:`;
- possíveis tags HTML restantes.

In [ ]:
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""

    texto = html.unescape(texto)

    texto = BeautifulSoup(
        texto,
        "lxml"
    ).get_text(" ", strip=True)

    texto = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        texto
    )

    texto = re.sub(
        r"\bH[1-6]\s*:\s*",
        " ",
        texto,
        flags=re.IGNORECASE
    )

    texto = re.sub(
        r"[\r\n\t]+",
        " ",
        texto
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    ).strip()

    return texto


df["texto_limpo"] = df["texto_bruto"].apply(limpar_texto)

df[["texto_bruto", "texto_limpo"]].head(3)

## 9. Preparação do ambiente

São usados:
- NLTK para tokenização, stopwords e stemming;
- spaCy para lematização.

A palavra `não` será mantida porque pode ser importante para entender o sentido de uma notícia.

In [ ]:
import os

os.environ["SSL_CERT_FILE"] = certifi.where()

for recurso in ["punkt", "punkt_tab", "stopwords", "rslp"]:
    print(f"Verificando {recurso}...")

    sucesso = nltk.download(recurso)

    if not sucesso:
        raise RuntimeError(
            f"Não foi possível baixar o recurso: {recurso}"
        )

try:
    nlp = spacy.load("pt_core_news_sm")

except OSError:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "spacy",
            "download",
            "pt_core_news_sm"
        ],
        check=True
    )

    nlp = spacy.load("pt_core_news_sm")

print("Ambiente preparado.")

## 10. Tokenização

In [ ]:
def tokenizar(texto):
    return word_tokenize(
        texto,
        language="portuguese"
    )


df["tokens"] = df["texto_limpo"].apply(tokenizar)

df[["titulo", "tokens"]].head(3)

## 11. Normalização

Os tokens são convertidos para minúsculas.

Também são mantidos apenas tokens formados por letras.

In [ ]:
def normalizar_tokens(tokens):
    return [
        token.casefold()
        for token in tokens
        if token.isalpha()
    ]


df["tokens_normalizados"] = df["tokens"].apply(
    normalizar_tokens
)

df[[
    "tokens",
    "tokens_normalizados"
]].head(3)

## 12. Remoção de stopwords

Stopwords são palavras muito frequentes, como artigos e preposições.

A palavra `não` foi retirada da lista de stopwords para não perder informação importante do texto.

In [ ]:
stopwords_pt = set(
    stopwords.words("portuguese")
)

stopwords_pt.discard("não")


def remover_stopwords(tokens):
    return [
        token
        for token in tokens
        if token not in stopwords_pt
    ]


df["tokens_sem_stopwords"] = df[
    "tokens_normalizados"
].apply(remover_stopwords)

df[[
    "tokens_normalizados",
    "tokens_sem_stopwords"
]].head(3)

## 13. Lematização

A lematização tenta transformar palavras flexionadas em uma forma base.

Exemplo: `ações` pode virar `ação`.

In [ ]:
def lematizar(tokens):
    if not tokens:
        return []

    documento = nlp(
        " ".join(tokens)
    )

    return [
        token.lemma_
        for token in documento
    ]


df["lemas"] = df[
    "tokens_sem_stopwords"
].apply(lematizar)

df[[
    "tokens_sem_stopwords",
    "lemas"
]].head(3)

## 14. Stemming

O stemming reduz as palavras a radicais.

O resultado nem sempre forma uma palavra existente, mas ajuda a aproximar palavras parecidas.

In [ ]:
stemmer = RSLPStemmer()


def aplicar_stemming(tokens):
    return [
        stemmer.stem(token)
        for token in tokens
    ]


df["stems"] = df[
    "tokens_sem_stopwords"
].apply(aplicar_stemming)

df[[
    "tokens_sem_stopwords",
    "stems"
]].head(3)

## 15. Comparação

A tabela abaixo permite comparar o texto original com as etapas de preparação.

In [ ]:
display(
    df[[
        "titulo",
        "texto_bruto",
        "texto_limpo",
        "tokens",
        "tokens_normalizados",
        "tokens_sem_stopwords",
        "lemas",
        "stems",
    ]].head(5)
)

## 16. Remoção de registros repetidos

São removidas notícias com a mesma URL.

Também são removidos títulos exatamente iguais.

In [ ]:
antes = len(df)

df = (
    df
    .drop_duplicates(subset=["url"])
    .drop_duplicates(subset=["titulo"])
    .reset_index(drop=True)
)

depois = len(df)

print("Antes:", antes)
print("Depois:", depois)
print("Removidos:", antes - depois)

## 17. Exportação

São gerados dois arquivos:
- uma base com os dados coletados;
- outra com todas as etapas de preparação.

In [ ]:
PASTA_SAIDA = Path("dados_pln")

PASTA_SAIDA.mkdir(
    exist_ok=True
)

colunas_brutas = [
    "url",
    "titulo",
    "data_publicacao",
    "autor",
    "texto_bruto",
    "tickers_encontrados",
]

df[colunas_brutas].to_csv(
    PASTA_SAIDA / "noticias_financeiras_brutas.csv",
    index=False,
    encoding="utf-8-sig",
)

df.to_csv(
    PASTA_SAIDA / "noticias_financeiras_processadas.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Arquivos gerados em:")
print(PASTA_SAIDA.resolve())

## 18. Atualização da base

A função abaixo permite juntar uma nova coleta com uma base já existente.

As URLs repetidas são removidas.

In [ ]:
def atualizar_historico(
    df_novo,
    caminho_csv
):
    caminho_csv = Path(
        caminho_csv
    )

    if caminho_csv.exists():
        antigo = pd.read_csv(
            caminho_csv
        )

        combinado = pd.concat(
            [antigo, df_novo],
            ignore_index=True
        )

    else:
        combinado = df_novo.copy()

    combinado = (
        combinado
        .drop_duplicates(
            subset=["url"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    combinado.to_csv(
        caminho_csv,
        index=False,
        encoding="utf-8-sig"
    )

    return combinado


# Exemplo:
# historico = atualizar_historico(
#     df[colunas_brutas],
#     PASTA_SAIDA / "historico_noticias.csv"
# )

## 19. Observações

Alguns pontos que podem afetar a coleta:
- mudanças na estrutura do site;
- notícias que citam várias empresas;
- notícias sem ticker escrito no texto;
- páginas que não tragam todos os campos esperados.

Por isso, a base precisa ser conferida depois da coleta.